### 🛠️ Phase 1: Delta Encoding & Decoding

Delta encoding is a lossy-to-lossless preprocessing technique widely used in data compression, especially effective for continuous data streams or gradient images.

#### 📌 Core Principle
Instead of storing absolute values, the encoder records the **difference (delta) between the current byte and the previous byte**.
* **Encoding Formula**: $D_i = (X_i - X_{i-1}) \pmod{256}$
* **Decoding Formula**: $X_i = (D_i + X_{i-1}) \pmod{256}$

#### 🌟 Design Highlight: Modular Arithmetic Defense
* **Overflow/Underflow Prevention**: By applying the `% 256` modulo operation, the data is **guaranteed to stay perfectly within the unsigned 8-bit byte range (`0 ~ 255`)** even when subtractions yield negative numbers or additions overflow. This completely eliminates hardware-level overflow bugs.

#### 🎯 Primary Objective
This preprocessing transforms varying data streams into **large clusters of repeating `0`s or uniform values**, drastically reducing the information entropy and smoothing the path for the subsequent LZ77 algorithm.

In [1]:
def delta_encode(data: bytes) -> bytearray:
    """Transforms continuous data into incremental byte-level differences (Deltas).
    
    This preprocessing layout reduces entropy by generating clusters of zeros,
    which optimizes the data stream for subsequent LZ77 window matching.
    """
    if not data:
        return bytearray()
    
    output = bytearray(len(data))
    output[0] = data[0] 
    
    for i in range(1, len(data)):
        # 🌟 WHY: Using modular arithmetic (% 256) ensures the delta wraps around seamlessly.
        # This keeps the result strictly within the unsigned 8-bit range (0-255) and prevents integer underflow.
        output[i] = (data[i] - data[i-1]) % 256
    return output

def delta_decode(data: bytearray) -> bytes:
    """Reconstructs the original byte stream from modulo-encoded delta differences."""
    if not data:
        return b""
    
    output = bytearray(len(data))
    output[0] = data[0]
    
    for i in range(1, len(data)):
        # 🌟 WHY: Reverses the encoding phase. Modulo 256 automatically resolves any previous underflow wraps.
        output[i] = (data[i] + output[i-1]) % 256
    return bytes(output)

### 📦 Phase 2: LZ77 Sliding Window Compression

This phase implements a Sliding Window dictionary-based compression algorithm, utilizing a Hash Map to accelerate repeated string matching.

#### ⚙️ Strict Boundary & Parameter Constraints
To ensure a seamless handoff to the downstream binary bit-packing stage, parameters are strictly capped to prevent bit-width overflow:
* **Search Window Size**: Rigidly locked at `3000` bytes. This ensures the offset value fits comfortably under the 12-bit maximum threshold of 4095, acting as a boundary defense.
* **Lookahead Buffer**: The maximum match length is strictly capped at `255` bytes, aligning perfectly with the 8-bit upper limit.

#### 🚀 Performance Optimization
* **Triple-Hash Acceleration**: A dictionary (`pos_hash`) tracks historical byte positions using a 3-byte tuple (`triple`) as the key. This drastically reduces the search complexity from $O(N^2)$ window scanning to near-constant lookups.

#### 📄 Output Token Format
The data stream is converted into a list of structurally unified tokens:
1. **Literal Token (No Match)**: `(False, literal_value, 0)`
2. **Reference Token (Match)**: `(True, distance, length)`

In [2]:
def lz77_compress(data: bytes, window_size: int = 3000) -> list:
    """Compresses data using a sliding window dictionary approach.
    
    Returns a list of structured tokens:
    - Literal: (False, byte_value, 0)
    - Reference Match: (True, distance, length)
    """
    tokens = []
    cursor = 0
    data_len = len(data)
    max_match_len = 255  # 🌟 WHY: Capped at 255 to perfectly fit into a single downstream 8-bit stream segment.
    
    pos_hash = {}
    
    while cursor < data_len:
        match_dist = 0
        match_len = 0
        
        if cursor + 3 <= data_len:
            triple = (data[cursor], data[cursor+1], data[cursor+2])
            p = pos_hash.get(triple, -1)
            
            # 🌟 WHY: Enforces boundary controls. The historical position must reside inside the active 3000-byte window.
            if p != -1 and (cursor - p <= window_size) and (p < cursor):
                curr_match_len = 0
                while (cursor + curr_match_len < data_len and \
                       data[p + curr_match_len] == data[cursor + curr_match_len] and \
                       curr_match_len < max_match_len):
                    curr_match_len += 1
                    
                if curr_match_len >= 3:
                    match_len = curr_match_len
                    match_dist = cursor - p

        if match_len >= 3:
            # 🌟 WHY: Defensive check to ensure values do not exceed the architectural limits of 12-bit/8-bit bitstreams.
            if 0 < match_dist <= 4095 and 3 <= match_len <= 255:
                tokens.append((True, match_dist, match_len))
                if cursor + 3 <= data_len:
                    triple = (data[cursor], data[cursor+1], data[cursor+2])
                    pos_hash[triple] = cursor
                cursor += match_len
                continue
                
        tokens.append((False, data[cursor], 0))
        if cursor + 3 <= data_len:
            triple = (data[cursor], data[cursor+1], data[cursor+2])
            pos_hash[triple] = cursor
        cursor += 1
            
    return tokens

def lz77_decompress(tokens: list) -> bytes:
    """Restores the raw byte sequences from LZ77 literal and reference tokens."""
    output = bytearray()
    for is_match, val, length in tokens:
        if not is_match:
            output.append(val)
        else:
            distance = val
            start_pos = len(output) - distance
            for i in range(length):
                output.append(output[start_pos + i])
    return bytes(output)

### 🌳 Phase 3: Dynamic Huffman Coding & Binary Bit-Packing

This stage handles the secondary compression of the LZ77 tokens and packs non-byte-aligned binary streams into actual storage files.

#### 🔣 Extended Alphabet & Special Symbol Definitions
To cleanly multiplex data literals and control signals into a single Huffman tree, custom symbols 256 and 257 are defined:
* `0 ~ 255`: Represent standard byte literal values.
* `256`: **End-of-File (EOF) Marker**. This signals the exact termination point during decompression, preventing the decoder from reading padding bits or trailing garbage data.
* `257`: **LZ77 Match Flag**. When the decoder encounters this symbol, it triggers a state machine to expect back-to-back reference metadata.

#### 🛡️ Bit-Packing Defensive Controls
* **Bit Mask Protection**: During writes, a bitwise mask `value & ((1 << num_bits) - 1)` is applied to forcefully sever any unintended bit overflow that could corrupt adjacent segments.
* **Compact Stream Layout**: Immediately following a `257` Match Flag, the encoder tightly packs a **strict 12-bit distance** and a **strict 8-bit length** into the stream to optimize spatial density.

In [3]:
import heapq
from collections import Counter
import pickle

class HuffmanNode:
    """Represents a node within the dynamic Huffman coding tree topology."""
    def __init__(self, symbol=None, freq=0):
        self.symbol = symbol
        self.freq = freq
        self.left = None
        self.right = None
        
    def __lt__(self, other):
        # Required by heapq to prioritize nodes with lower frequencies during tree consolidation
        return self.freq < other.freq

class BitWriter:
    """Manages an un-aligned bitstream, packing arbitrary bit-widths into a standard bytearray."""
    def __init__(self):
        self.bytes_data = bytearray()
        self.buffer = 0
        self.bit_count = 0

    def write_bits(self, value: int, num_bits: int):
        # 🌟 WHY: Architectural Mask Defense. Forcefully cuts off high-order bits to prevent unintended data spillover.
        value = value & ((1 << num_bits) - 1)
        for i in range(num_bits - 1, -1, -1):
            bit = (value >> i) & 1
            self.buffer = (self.buffer << 1) | bit
            self.bit_count += 1
            if self.bit_count == 8:
                self.bytes_data.append(self.buffer)
                self.buffer = 0
                self.bit_count = 0

    def flush(self) -> bytearray:
        """Flushes remaining bits left in the active buffer by left-aligning them into a final byte."""
        if self.bit_count > 0:
            self.buffer = self.buffer << (8 - self.bit_count)
            self.bytes_data.append(self.buffer)
            self.buffer = 0
            self.bit_count = 0
        return self.bytes_data

class BitReader:
    """Parses arbitrary sequential bit lengths from a packed binary payload."""
    def __init__(self, data: bytes):
        self.data = data
        self.byte_idx = 0
        self.bit_idx = 7

    def read_bit(self) -> int:
        if self.byte_idx >= len(self.data):
            return 0
        bit = (self.data[self.byte_idx] >> self.bit_idx) & 1
        self.bit_idx -= 1
        if self.bit_idx < 0:
            self.bit_idx = 7
            self.byte_idx += 1
        return bit

    def read_bits(self, num_bits: int) -> int:
        value = 0
        for _ in range(num_bits):
            value = (value << 1) | self.read_bit()
        return value

def build_huffman_tree_root(frequencies: dict) -> HuffmanNode:
    """Assembles a Huffman tree using a min-heap queue based on symbol frequencies."""
    if len(frequencies) == 0:
        frequencies = {256: 1} # Fallback to prevent crash on empty datasets
    if len(frequencies) == 1:
        # Heap requires at least two distinct nodes to initiate merging loops
        k = list(frequencies.keys())[0]
        frequencies[k+1 if k < 257 else k-1] = 1
        
    heap = [HuffmanNode(sym, freq) for sym, freq in frequencies.items()]
    heapq.heapify(heap)
    
    while len(heap) > 1:
        node1 = heapq.heappop(heap)
        node2 = heapq.heappop(heap)
        merged = HuffmanNode(freq=node1.freq + node2.freq)
        merged.left = node1
        merged.right = node2
        heapq.heappush(heap, merged)
    return heap[0]

def generate_codes_from_tree(root: HuffmanNode, current_code="", code_table=None) -> dict:
    """Recursively walks the Huffman tree to map symbols to their variable-length binary strings."""
    if code_table is None:
        code_table = {}
    if root is None:
        return code_table
    if root.symbol is not None:
        code_table[root.symbol] = current_code
        return code_table
    generate_codes_from_tree(root.left, current_code + "0", code_table)
    generate_codes_from_tree(root.right, current_code + "1", code_table)
    return code_table

## 💾 Custom File Layout (.MY) and End-to-End Codec

This section integrates all the independent components into a coherent, production-ready `my_custom_compress` and `my_custom_decompress` pipeline.

### 📊 Binary File Layout

The final compressed file structure under the `.MY` format is explicitly laid out as follows:

| Field Name | Length / Format | Description |
| :--- | :--- | :--- |
| **Magic Number** | 2 Bytes (`b"MY"`) | Identifies the file as a valid proprietary compressed format |
| **Original Size** | 4 Bytes Unsigned Int | Total byte size of the uncompressed source file (Little-Endian) |
| **Tree Length** | 4 Bytes Unsigned Int | Byte length of the serialized Huffman tree (Little-Endian) |
| **Huffman Tree** | Dynamic (Pickle Serialized) | The topology structure required to reconstruct the decoder |
| **Compressed Bits**| Dynamic (Binary Stream) | The payload consisting of packed Huffman codes and LZ77 metadata |

### 🛡️ Decompression End "Ultimate Fault-Tolerance"
In `my_custom_decompress`, rigorous fallback logic is implemented to handle corrupted bitstreams or edge-case anomalies defensively:
* **Distance Clamping**: If the decoded distance exceeds the current output bounds (`distance > len(output)`), it is safely clamped to prevent an abrupt `IndexError` crash.
* **Infinite Loop Prevention**: If the distance unrolls to `0`, the decoder injects a zero-byte and safely bypasses the block to avoid getting trapped in an infinite pointer stall.

In [4]:
import struct

def my_custom_compress(original_data: bytes) -> bytes:
    """Encodes raw data through Delta transformation, LZ77, and Huffman bit-packing.
    
    Returns a unified binary payload prefixed with custom header metadata.
    """
    if not original_data:
        return b""
    
    delta_data = delta_encode(original_data)
    lz77_tokens = lz77_compress(delta_data)
    
    symbols = []
    frequencies = Counter()
    for is_match, val, length in lz77_tokens:
        if not is_match:
            symbols.append((val, None, None))
            frequencies[val] += 1
        else:
            # 🌟 WHY: Multiplexing tokens. '257' acts as an in-stream control signal indicating an incoming match block.
            symbols.append((257, val, length))
            frequencies[257] += 1
            
    # 🌟 WHY: Symbol '256' serves as an explicit EOF flag to prevent the decoder from over-reading trailing padding bits.
    symbols.append((256, None, None))
    frequencies[256] += 1
    
    root_node = build_huffman_tree_root(frequencies)
    huff_table = generate_codes_from_tree(root_node)
    
    writer = BitWriter()
    for sym, dist, length in symbols:
        bit_str = huff_table[sym]
        writer.write_bits(int(bit_str, 2), len(bit_str))
        if sym == 257:
            writer.write_bits(dist, 12)   # Constrained to 12 bits matching sliding window configuration
            writer.write_bits(length, 8)  # Constrained to 8 bits matching lookahead buffer configuration
            
    compressed_bits = writer.flush()
    
    # TODO: Migrate from pickle format to Canonical Huffman table dump to lower small-file header overhead.
    tree_bytes = pickle.dumps(root_node)
    
    # Layout arrangement: Magic Number (2B), Original Size (4B), Tree Length (4B)
    header = struct.pack("<2sI", b"MY", len(original_data)) + struct.pack("<I", len(tree_bytes))
    return header + tree_bytes + compressed_bits

def my_custom_decompress(compressed_bytes: bytes) -> bytes:
    """Decodes custom binary stream back into its original uncompressed data layout."""
    if not compressed_bytes:
        return b""
        
    magic, orig_size = struct.unpack("<2sI", compressed_bytes[:6])
    if magic != b"MY":
        raise ValueError("Invalid custom compressed file layout configuration.")
        
    tree_len = struct.unpack("<I", compressed_bytes[6:10])[0]
    root_node = pickle.loads(compressed_bytes[10:10+tree_len])
    
    bit_data = compressed_bytes[10+tree_len:]
    reader = BitReader(bit_data)
    output = bytearray()
    
    while True:
        curr_node = root_node
        while curr_node.symbol is None:
            bit = reader.read_bit()
            curr_node = curr_node.left if bit == 0 else curr_node.right
                
        sym = curr_node.symbol
        
        if sym == 256:
            break
        elif sym <= 255:
            output.append(sym)
        elif sym == 257:
            distance = reader.read_bits(12)
            length = reader.read_bits(8)
            
            # 🌟 WHY: Fault-Tolerance Protection. If the bitstream is malformed, clamp distance 
            # to prevent runtime IndexError terminations.
            if distance > len(output):
                distance = len(output)
            if distance == 0:
                # 🌟 WHY: Dead-loop counter-measure. Infuses a safe baseline if distance collapses to 0.
                for _ in range(length):
                    output.append(0)
                continue
                
            start_pos = len(output) - distance
            for _ in range(length):
                output.append(output[start_pos])
                start_pos += 1
                        
    return delta_decode(output)

In [6]:
import os
import time
import datetime
import numpy as np
import pandas as pd
from collections import Counter

# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================
# Change these variables every time you upgrade your algorithm
CURRENT_VERSION = "v1_base"
VERSION_NOTES = "Initial version: Delta + LZ77(3000,255) + Huffman (Pickle tree)"

# Benchmark settings
HISTORY_FILE = "benchmark_history.csv"
NUM_RUNS = 5  # Number of runs to average out timing noise
TARGET_FILES = [
    "test1.txt",
    "test2.txt",
    "test3.txt",
    "Lenna.bmp",
    "Cameraman.bmp"
]

def run_benchmark_with_history():
    """
    Runs the benchmark for the current version, persists the data into a CSV file,
    and displays both the current run and historical comparison.
    """
    current_results = []
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    print(f"🚀 Starting Benchmark for Version: {CURRENT_VERSION}")
    print(f"📝 Notes: {VERSION_NOTES}")
    print(f"🔄 Repeating each test {NUM_RUNS} times to eliminate timing noise...\n")
    
    for file_name in TARGET_FILES:
        if not os.path.exists(file_name):
            print(f"⚠️ Warning: '{file_name}' not found. Skipped.")
            continue
            
        # 1. Read original data
        with open(file_name, "rb") as f:
            original_data = f.read()
            
        orig_size = len(original_data)
        if orig_size == 0:
            continue
            
        # 2. Benchmark Compression (Run multiple times for averaging)
        comp_times = []
        compressed_data = b""
        for _ in range(NUM_RUNS):
            t0 = time.perf_counter()
            compressed_data = my_custom_compress(original_data)
            t1 = time.perf_counter()
            comp_times.append((t1 - t0) * 1000) # Convert to ms
            
        comp_time_ms = np.mean(comp_times)
        comp_size = len(compressed_data)
        
        # 3. Benchmark Decompression (Run multiple times for averaging)
        decomp_times = []
        decompressed_data = b""
        for _ in range(NUM_RUNS):
            t2 = time.perf_counter()
            decompressed_data = my_custom_decompress(compressed_data)
            t3 = time.perf_counter()
            decomp_times.append((t3 - t2) * 1000) # Convert to ms
            
        decomp_time_ms = np.mean(decomp_times)
        
        # 4. Calculate Objective Metrics
        comp_ratio = orig_size / comp_size if comp_size > 0 else 0
        space_saving = ((orig_size - comp_size) / orig_size) * 100
        
        # Throughput in MB/s = (Bytes / 1024 / 1024) / (ms / 1000)
        orig_size_mb = orig_size / (1024 * 1024)
        comp_throughput = orig_size_mb / (comp_time_ms / 1000) if comp_time_ms > 0 else 0
        decomp_throughput = orig_size_mb / (decomp_time_ms / 1000) if decomp_time_ms > 0 else 0
        
        is_valid = "PASS" if decompressed_data == original_data else "FAIL"
        
        # 5. Store current run data
        current_results.append({
            "timestamp": timestamp,
            "version": CURRENT_VERSION,
            "notes": VERSION_NOTES,
            "file_name": file_name,
            "orig_size_bytes": orig_size,
            "comp_size_bytes": comp_size,
            "comp_time_ms": round(comp_time_ms, 2),
            "decomp_time_ms": round(decomp_time_ms, 2),
            "comp_ratio": round(comp_ratio, 2),
            "space_saving_pct": round(space_saving, 2),
            "comp_throughput_mbs": round(comp_throughput, 2),
            "decomp_throughput_mbs": round(decomp_throughput, 2),
            "verification": is_valid
        })

    # Create DataFrame for the current run
    df_current = pd.DataFrame(current_results)
    
    # ==========================================
    # PERSISTENCE (SAVE TO CSV)
    # ==========================================
    if os.path.exists(HISTORY_FILE):
        df_history = pd.read_csv(HISTORY_FILE)
        # Prevent appending duplicate entries if the block is re-run with the same version on the exact same files
        # We drop existing logs for the same version and file to keep the history clean
        df_history = df_history[~((df_history["version"] == CURRENT_VERSION) & (df_history["file_name"].isin(df_current["file_name"])))]
        df_new_history = pd.concat([df_history, df_current], ignore_index=True)
    else:
        df_new_history = df_current

    df_new_history.to_csv(HISTORY_FILE, index=False)
    print(f"💾 Successfully saved and updated results in '{HISTORY_FILE}'.")
    
    # ==========================================
    # DISPLAY 1: Current Run Report (Formatted)
    # ==========================================
    print("\n📊 --- CURRENT RUN REPORT ---")
    display_df = df_current.copy()
    display_df["orig_size_bytes"] = display_df["orig_size_bytes"].map("{:,}".format)
    display_df["comp_size_bytes"] = display_df["comp_size_bytes"].map("{:,}".format)
    display_df["comp_ratio"] = display_df["comp_ratio"].map("{:.2f}x".format)
    display_df["space_saving_pct"] = display_df["space_saving_pct"].map("{:.2f}%".format)
    display(display_df[[
        "file_name", "orig_size_bytes", "comp_size_bytes", 
        "comp_time_ms", "decomp_time_ms", "comp_ratio", 
        "space_saving_pct", "comp_throughput_mbs", "decomp_throughput_mbs", "verification"
    ]])
    
    # ==========================================
    # DISPLAY 2: Cross-Version Historical Comparison
    # ==========================================
    print("\n📈 --- HISTORICAL VERSION COMPARISON (Pivot Table) ---")
    # Reload full history to ensure data integrity
    df_full_history = pd.read_csv(HISTORY_FILE)
    
    # Pivot table to compare Space Saving (%) across versions for each file
    pivot_space = df_full_history.pivot_table(
        index="file_name", 
        columns="version", 
        values="space_saving_pct"
    )
    
    # Sort index to match target files order for consistency
    existing_targets = [f for f in TARGET_FILES if f in pivot_space.index]
    pivot_space = pivot_space.reindex(existing_targets)
    
    print("\n[Metric: Space Saving (%)] -> Higher is better. Negative means file expansion.")
    display(pivot_space.style.format("{:.2f}%").highlight_max(axis=1, color="lightgreen"))

# Execute the benchmark
run_benchmark_with_history()

🚀 Starting Benchmark for Version: v1_base
📝 Notes: Initial version: Delta + LZ77(3000,255) + Huffman (Pickle tree)
🔄 Repeating each test 5 times to eliminate timing noise...

💾 Successfully saved and updated results in 'benchmark_history.csv'.

📊 --- CURRENT RUN REPORT ---


,file_name,orig_size_bytes,comp_size_bytes,comp_time_ms,decomp_time_ms,comp_ratio,space_saving_pct,comp_throughput_mbs,decomp_throughput_mbs,verification
0,test1.txt,35,"1,422",0.18,0.09,0.02x,-3962.86%,0.19,0.39,PASS
1,test2.txt,"2,638","7,951",5.91,3.53,0.33x,-201.40%,0.43,0.71,PASS
2,test3.txt,"5,349","10,865",7.01,6.81,0.49x,-103.12%,0.73,0.75,PASS
3,Lenna.bmp,"263,224","225,308",482.30,415.34,1.17x,14.40%,0.52,0.60,PASS
4,Cameraman.bmp,"66,616","68,288",100.45,109.38,0.98x,-2.51%,0.63,0.58,PASS



📈 --- HISTORICAL VERSION COMPARISON (Pivot Table) ---

[Metric: Space Saving (%)] -> Higher is better. Negative means file expansion.


version,v1_base
file_name,
test1.txt,-3962.86%
test2.txt,-201.40%
test3.txt,-103.12%
Lenna.bmp,14.40%
Cameraman.bmp,-2.51%
